In [1]:
import pandas as pandas
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = pandas.read_csv('data.csv')
data.head()

,32.502345269453031,31.70700584656992
0,53.426804,68.777596
1,61.530358,62.562382
2,47.475640,71.546632
3,59.813208,87.230925
4,55.142188,78.211518


In [5]:
data.shape

(99, 2)

In [6]:
data.describe()

,32.502345269453031,31.70700584656992
count,99.000000,99.000000
mean,49.124564,73.149475
std,9.652463,16.216558
min,25.128485,41.412885
25%,41.648159,61.088576
50%,50.030174,72.247251
75%,56.798054,83.287411
max,70.346076,118.591217


In [8]:
import pandas as pd

# Column names
cols = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "median_house_value"
]

# Load the file
df = pd.read_csv("cal_housing.data", names=cols)

# Preview
print(df.head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value  
0       322.0       126.0         8.3252            452600.0  
1      2401.0      1138.0         8.3014            358500.0  
2       496.0       177.0         7.2574            352100.0  
3       558.0       219.0         5.6431            341300.0  
4       565.0       259.0         3.8462            342200.0  


In [9]:
df.shape

(20640, 9)

In [10]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.898014,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.247906,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,295.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [11]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
import xgboost as xgb
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
import shap


In [12]:
# ─── Load Data ────────────────────────────────────────────────────────────────
df = pd.read_csv('/home/claude/cal_housing.csv')

print("Dataset shape:", df.shape)
print(df.dtypes)
print(df.isnull().sum())


FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/cal_housing.csv'

In [ ]:

# ─── Load Data ────────────────────────────────────────────────────────────────
df = pd.read_csv('/home/claude/cal_housing.csv')

print("Dataset shape:", df.shape)
print(df.dtypes)
print(df.isnull().sum())

# ─── Feature Engineering ──────────────────────────────────────────────────────
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room']   = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

# Log transform skewed features
for col in ['total_rooms','total_bedrooms','population','households']:
    df[f'log_{col}'] = np.log1p(df[col])

# Interaction: income x coastal proxy
df['income_x_coastal'] = df['median_income'] * np.exp(-0.3*(df['longitude']+122)**2)

# Target log transform
df['log_value'] = np.log(df['median_house_value'])

features_base = ['longitude','latitude','housing_median_age','median_income',
                 'rooms_per_household','bedrooms_per_room','population_per_household']
features_eng  = features_base + ['income_x_coastal','log_total_rooms','log_households']

X = df[features_eng]
y = df['log_value']
y_raw = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
_, _, y_raw_train, y_raw_test = train_test_split(X, y_raw, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

results = {}

# ─── 1. OLS Linear Regression ────────────────────────────────────────────────
ols = LinearRegression()
ols.fit(X_train_s, y_train)
pred_ols = ols.predict(X_test_s)
rmse_ols = np.sqrt(mean_squared_error(y_test, pred_ols))
r2_ols   = r2_score(y_test, pred_ols)
results['OLS Linear'] = {'rmse': rmse_ols, 'r2': r2_ols,
                          'pred': np.exp(pred_ols), 'actual': y_raw_test.values}
print(f"OLS: RMSE={rmse_ols:.4f}, R2={r2_ols:.4f}")

# ─── 2. Ridge Regression ─────────────────────────────────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)
pred_ridge = ridge.predict(X_test_s)
results['Ridge'] = {'rmse': np.sqrt(mean_squared_error(y_test, pred_ridge)),
                     'r2':   r2_score(y_test, pred_ridge),
                     'pred': np.exp(pred_ridge), 'actual': y_raw_test.values}
print(f"Ridge: RMSE={results['Ridge']['rmse']:.4f}, R2={results['Ridge']['r2']:.4f}")

# ─── 3. Random Forest ────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=5,
                            n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
results['Random Forest'] = {'rmse': np.sqrt(mean_squared_error(y_test, pred_rf)),
                              'r2':   r2_score(y_test, pred_rf),
                              'pred': np.exp(pred_rf), 'actual': y_raw_test.values}
print(f"Random Forest: RMSE={results['Random Forest']['rmse']:.4f}, R2={results['Random Forest']['r2']:.4f}")

# ─── 4. Gradient Boosting ────────────────────────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                 subsample=0.8, random_state=42)
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)
results['Gradient Boosting'] = {'rmse': np.sqrt(mean_squared_error(y_test, pred_gb)),
                                  'r2':   r2_score(y_test, pred_gb),
                                  'pred': np.exp(pred_gb), 'actual': y_raw_test.values}
print(f"GB: RMSE={results['Gradient Boosting']['rmse']:.4f}, R2={results['Gradient Boosting']['r2']:.4f}")

# ─── 5. XGBoost ──────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                               subsample=0.8, colsample_bytree=0.8,
                               reg_alpha=0.1, reg_lambda=1.0,
                               random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
pred_xgb = xgb_model.predict(X_test)
results['XGBoost'] = {'rmse': np.sqrt(mean_squared_error(y_test, pred_xgb)),
                       'r2':   r2_score(y_test, pred_xgb),
                       'pred': np.exp(pred_xgb), 'actual': y_raw_test.values}
print(f"XGBoost: RMSE={results['XGBoost']['rmse']:.4f}, R2={results['XGBoost']['r2']:.4f}")

# ─── 6. LOWESS (Nonparametric) ────────────────────────────────────────────────
# Fit on income -> log_value (1D LOWESS for illustration)
income_all = df['median_income'].values
lv_all     = df['log_value'].values
sort_idx   = np.argsort(income_all)
lowess_fit = lowess(lv_all[sort_idx], income_all[sort_idx], frac=0.2, return_sorted=True)
np.save('/home/claude/lowess_fit.npy', lowess_fit)
print("LOWESS done")

# ─── 7. Polynomial / Spline via statsmodels ──────────────────────────────────
# Fit OLS with cubic polynomial of income + location
import statsmodels.formula.api as smf
df_sm = df[['log_value','median_income','latitude','longitude',
            'housing_median_age','rooms_per_household','population_per_household']].copy()
formula = ("log_value ~ median_income + I(median_income**2) + I(median_income**3) "
           "+ latitude + longitude + I(latitude**2) + I(longitude**2) "
           "+ housing_median_age + rooms_per_household + population_per_household")
model_poly = smf.ols(formula, data=df_sm).fit()
print(model_poly.summary().tables[0])

# In-sample R2 for polynomial model
idx_test = y_test.index
pred_poly = model_poly.predict(df_sm.loc[idx_test])
results['Polynomial OLS'] = {
    'rmse': np.sqrt(mean_squared_error(y_test, pred_poly)),
    'r2':   r2_score(y_test, pred_poly),
    'pred': np.exp(pred_poly.values), 'actual': y_raw_test.values
}
print(f"Poly OLS: RMSE={results['Polynomial OLS']['rmse']:.4f}, R2={results['Polynomial OLS']['r2']:.4f}")

# ─── 8. GAM via pygam ────────────────────────────────────────────────────────
from pygam import LinearGAM, s, f, l
X_gam = X[features_eng].values
y_gam = y.values
Xtr_g = X_gam[X_train.index]
Xte_g = X_gam[X_test.index]
ytr_g = y_gam[X_train.index]
yte_g = y_gam[X_test.index]

gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9),
                n_splines=10).fit(Xtr_g, ytr_g)
pred_gam = gam.predict(Xte_g)
results['GAM (Splines)'] = {
    'rmse': np.sqrt(mean_squared_error(yte_g, pred_gam)),
    'r2':   r2_score(yte_g, pred_gam),
    'pred': np.exp(pred_gam), 'actual': y_raw_test.values
}
print(f"GAM: RMSE={results['GAM (Splines)']['rmse']:.4f}, R2={results['GAM (Splines)']['r2']:.4f}")

# Save results
import pickle
with open('/home/claude/results.pkl', 'wb') as f_:
    pickle.dump(results, f_)

# ─── 9. SHAP values for XGBoost ──────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test.iloc[:500])
np.save('/home/claude/shap_vals.npy', shap_vals)
shap_feat_names = list(X_test.columns)
with open('/home/claude/shap_feat_names.pkl','wb') as f_:
    pickle.dump(shap_feat_names, f_)

print("All models done. Saving metrics summary...")
summary = {k: {'rmse': v['rmse'], 'r2': v['r2']} for k,v in results.items()}
import json
with open('/home/claude/model_summary.json','w') as f_:
    json.dump(summary, f_, indent=2)
print(json.dumps(summary, indent=2))
PYEOF
python3 /home/claude/run_analysis.py
Output

Dataset shape: (20640, 9)
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
dtype: object
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64
OLS: RMSE=0.4129, R2=0.6960
Ridge: RMSE=0.4129, R2=0.6960
Random Forest: RMSE=0.2617, R2=0.8780
GB: RMSE=0.2563, R2=0.8829
XGBoost: RMSE=0.2564, R2=0.8828
LOWESS done
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              log_value   R-squared:                       0.783
Model:                            OLS   Adj. R-squared:                  0.783
Method:                 Least Squares   F-statistic:                     7440.
Date:                Mon, 11 May 2026   Prob (F-statistic):               0.00
Time:                        15:25:15   Log-Likelihood:                -7796.6
No. Observations:               20640   AIC:                         1.562e+04
Df Residuals:                   20629   BIC:                         1.570e+04
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================
Poly OLS: RMSE=0.3504, R2=0.7812
GAM: RMSE=0.3243, R2=0.8125
All models done. Saving metrics summary...
{
  "OLS Linear": {
    "rmse": 0.4129396671590975,
    "r2": 0.6960362810073646
  },
  "Ridge": {
    "rmse": 0.41294009963970757,
    "r2": 0.6960356443115279
  },
  "Random Forest": {
    "rmse": 0.2616596492947986,
    "r2": 0.8779544208499578
  },
  "Gradient Boosting": {
    "rmse": 0.25633208713412525,
    "r2": 0.8828736822392637
  },
  "XGBoost": {
    "rmse": 0.25644133558633797,
    "r2": 0.8827738227494556
  },
  "Polynomial OLS": {
    "rmse": 0.350368050983247,
    "r2": 0.7811746976280383
  },
  "GAM (Splines)": {
    "rmse": 0.3243277593320885,
    "r2": 0.812493301952149
  }
}